# Inspect the training stack

This notebook uses the same data folders and helpers as `scripts/train.py`.
It adds plots so you can check the rasters and the spatial split before a long run.

To recreate training with no figures, use:

```bash
poetry run python scripts/train.py
```

Run this notebook from the Poetry environment (`poetry install --with dev`).
Set `data_root` in `config/paths.yaml` or `PREDICTPS_DATA_ROOT` first.
The last cell trains on a GPU. Skip it if you only want the plots.


In [ ]:
import os
import shutil
import time
import datetime

import torch
from torchgeo.datasets import BoundingBox

from ps_density.model_setup import training_setup
from ps_density.paths import load_paths
from ps_density.sampling_maps import plot_sampling_map
from ps_density.torchgeo_fun_to_read_input import (
    RasterDataset_imgs,
    RasterDataset_msks,
    plot_input,
    scale_coh,
)

_cfg = load_paths()
path_prefix = str(_cfg.outputs.training_dir)
experiment_name = _cfg.outputs.training_experiment_name
experiment_dir = os.path.join(path_prefix, experiment_name)
os.makedirs(experiment_dir, exist_ok=True)
print(experiment_dir)

imgs = RasterDataset_imgs(
    paths=str(_cfg.input.coherence_summer_dir), transforms=scale_coh
)
imgs_OSM = RasterDataset_imgs(paths=str(_cfg.input.osm_train_dir))
msks = RasterDataset_msks(paths=str(_cfg.input.mask_egms_dir))
dataset = (imgs & imgs_OSM) & msks

print("crs", dataset.crs, "res", dataset.res, "length", len(dataset))


In [ ]:
bounding_box = BoundingBox(
    minx=3.0,
    maxx=7.999999996,
    miny=50.000000004,
    maxy=54.0,
    mint=0.0,
    maxt=9.223372036854776e18,
)
sample = dataset[bounding_box]
print("input channels", sample["image"].shape[0])
print("mask channels", sample["mask"].shape[0])
print("mask values", torch.unique(sample["mask"]))
plot_input(sample, rows=1)


## Sampling map

Blue chips are one training batch, red one validation batch, green one test batch.
The split is 70 / 15 / 15 with the seed below. Defaults match the old debug map
(`train_batch=50`). Lower those indices if you only want a quick look.
The OSM basemap needs a network connection. The chip outlines are still drawn if it fails.


In [ ]:
seed = 54321
patch_size = 64
batch_size = 16

plot_sampling_map(
    dataset,
    seed=seed,
    patch_size=patch_size,
    batch_size=batch_size,
    train_batch=50,
    val_batch=5,
    test_batch=20,
    save_path=os.path.join(experiment_dir, "fig2.jpg"),
)


## Optional training

Same loop as `scripts/train.py`. Change `seed` or `param_opts` here, then run the cell.
Skip this cell if you only wanted the plots. Training needs a CUDA GPU.


In [ ]:
in_channels = 2
ignore_index = None
lr_patience = 10
min_delta_lr = 0.01
patience = 15
min_delta = 0.01
max_epochs = 300
gpu_id = 0
device = torch.device(f"cuda:{gpu_id}")
num_dataloader_workers = 0

param_const = [
    in_channels,
    ignore_index,
    lr_patience,
    min_delta_lr,
    patience,
    min_delta,
    max_epochs,
    gpu_id,
    device,
    num_dataloader_workers,
]
param_opts = [
    [16, 64, None, "resnet34", None, "mae", 0.00776601, 0.9, 0.99, 0.01],
]

for seed in [54321]:
    for i in range(len(param_opts)):
        trainer, task, datamodule = training_setup(
            seed, param_const, param_opts[i], dataset, experiment_dir, experiment_name
        )
        start_time = time.time()
        trainer.fit(model=task, datamodule=datamodule)
        print("Training took {:.2f} seconds".format(time.time() - start_time))
        new_name = "seed_{}_{}.ckpt".format(
            seed, datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        )
        shutil.copy(
            trainer.checkpoint_callback.best_model_path,
            os.path.join(experiment_dir, new_name),
        )

print("Training done")
